In [ ]:
%cd ..


In [ ]:
from pathlib import Path
import os
import h5py
import scipy.io


In [ ]:


path = "data/ninapro_db5/S2_E2_A1.mat"

mat = scipy.io.loadmat(path)
print(mat.keys())

In [ ]:
print(mat['emg'].shape)  # (number of samples, number of electrodes)
print(mat['acc'].shape)
print(mat['stimulus'].shape)  # (number of samples, 1)
print(mat['glove'].shape)
print(mat['subject'])
print(mat['exercise'])
print(mat['repetition'].shape)
print(mat['frequency'])
print(mat['sensor'])

In [ ]:
import numpy as np
emg = mat['emg']  # (number of samples, number of electrodes)
emg = np.array(emg, dtype=np.float32)
print(np.max(np.abs(emg)))  # Check the max absolute value for normalization

In [ ]:
stimulus = mat['stimulus']
repetition = mat['repetition']

print(np.unique(stimulus))  # Unique gesture labels
print(np.unique(repetition))  # Unique repetition numbers

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(25, 10))
plt.plot(stimulus, label='Stimulus (Gesture ID)')
plt.plot(repetition, label='Repetition')
plt.xlabel('Sample Index')
plt.ylabel('Value')
plt.title('Stimulus and Repetition Over Time')
plt.legend()
plt.show()

In [ ]:
restimulus = mat['restimulus']
rerepetition = mat['rerepetition']

print(np.unique(restimulus))  # Unique refined gesture labels
print(np.unique(rerepetition))  # Unique refined repetition numbers

In [ ]:
plt.figure(figsize=(25, 10))
plt.plot(restimulus, label='Refined Stimulus (Gesture ID)')
plt.plot(rerepetition, label='Refined Repetition')
plt.xlabel('Sample Index')
plt.ylabel('Value')
plt.title('Refined Stimulus and Repetition Over Time')
plt.legend()
plt.show()

## Check if the masking and segmenting script works with visualisation

In [ ]:
import h5py
from tqdm import tqdm



def get_global_gesture_id(exercise_id, stimulus_id):
    """NinaPro DB5 has a unique gesture distribution across exercises:
    
    Each exercise has a different set of gestures:

    Exercise 1 (2 in fname, B in docs): 17 gestures (1-17)
    Exercise 2 (3 in fname, C in docs): 23 gestures (18-40)
    Exercise 3 (1 in fname, A in docs): 12 gestures (41-52)
    """

    if stimulus_id == 0: return 0 # Rest is always 0
    
    # Offsets based on the 17, 23, 12 distribution
    if exercise_id == 1:
        return stimulus_id
    elif exercise_id == 2:
        return stimulus_id + 17
    elif exercise_id == 3:
        return stimulus_id + 17 + 23
    return stimulus_id

dataset_dir = "data/ninapro_db5"
output_file = "test.hdf5"

dataset_dir = Path(dataset_dir)
mat_files = sorted(dataset_dir.glob("*.mat"))

with h5py.File(output_file, 'w') as f:
    root = f.create_group("ninapro_db5")
    
    for fpath in tqdm(mat_files, desc="Converting NinaPro DB5"):
        mat = scipy.io.loadmat(str(fpath))
        
        # Extract basic subject info
        # Matlab stores strings/scalars in nested arrays; we flatten them
        subject_id = int(mat['subject'][0][0])
        exercise_id = int(mat['exercise'][0][0])
        fs = int(mat['frequency'][0][0])
        
        subj_key = f"S{subject_id}"
        if subj_key not in root:
            subj_group = root.create_group(subj_key)
            # Static Metadata from the website spec
            subj_group.attrs['age'] = int(mat['age'][0][0])
            subj_group.attrs['gender'] = str(mat['gender'][0])
            subj_group.attrs['weight_kg'] = float(mat['weight'][0][0])
            subj_group.attrs['height_cm'] = float(mat['height'][0][0])
            subj_group.attrs['laterality'] = str(mat['laterality'][0])
            subj_group.attrs['circumference_cm'] = float(mat['circumference'][0][0])
        else:
            subj_group = root[subj_key]

        # Continuous data arrays
        emg = mat['emg'].astype(np.float32)       # (T, 16)
        acc = mat['acc'].astype(np.float32)       # (T, 3) - Note: Only 3 cols for 1st Myo
        glove = mat['glove'].astype(np.float32)   # (T, 22)

        emg /= 128.0  # Normalize Myo EMG to [-1, 1] if raw values are in [-128, 127]
        
        # Use 'restimulus' and 'rerepetition' for refined labels
        stimulus = mat['restimulus'].flatten()
        repetition = mat['rerepetition'].flatten()
        
        # Group by refined trial (rerepetition)
        unique_reps = np.unique(repetition)
        unique_stimuli = np.unique(stimulus)
        print(unique_stimuli)
        i = 0
        
        for rep_idx in unique_reps:
            if rep_idx == 0: continue # Skip rest periods
            
            for stim_idx in unique_stimuli:
                if stim_idx == 0: continue # Skip rest periods
            
                mask = ((repetition == rep_idx) & (stimulus == stim_idx))
                if not np.any(mask):
                    continue
                gesture_ids = np.unique(stimulus[mask])
                print(gesture_ids)
                print(rep_idx)
                print(stim_idx)

                plt.figure(figsize=(25, 10))
                plt.plot(stimulus, label='Refined Stimulus (Gesture ID)')
                plt.plot(repetition, label='Refined Repetition')
                plt.plot(mask.astype(int), label='Current Repetition Mask')  # Highlight current repetition
                plt.xlabel('Sample Index')
                plt.ylabel('Value')
                plt.title('Refined Stimulus and Repetition Over Time')
                plt.legend()
                plt.show()
                i += 1
                if i > 3:
                    break
                continue
                
                # Filter out the '0' label within a repetition mask if it exists
                gesture_ids = gesture_ids[gesture_ids != 0]
                if len(gesture_ids) == 0: continue
                
                gesture_id = int(gesture_ids[0])
                gesture_id = get_global_gesture_id(exercise_id, gesture_id)

                # Create sample group (E=Exercise, R=Repetition, G=Gesture)
                sample_name = f"E{exercise_id}_R{rep_idx}_G{gesture_id}"
                sample_group = subj_group.create_group(sample_name)
                
                # Per-sample Metadata
                sample_group.attrs['gesture_id'] = gesture_id
                sample_group.attrs['repetition_id'] = rep_idx
                
                # 1. EMG (T, 16) 
                # NinaPro EMG is typically recorded in mV or raw units. 
                # Check if you need to scale. If raw Myo, use emg /= 128.0
                sample_group.create_dataset("emg", data=emg[mask], compression="lzf")
                
                # 2. Accel (T, 3)
                sample_group.create_dataset("accel", data=acc[mask], compression="lzf")
                
                # 3. Glove (T, 22) - Kinematic ground truth
                sample_group.create_dataset("glove", data=glove[mask], compression="lzf")
        

In [66]:
with h5py.File("data/ninapro_db5/ninapro_db5.hdf5", 'r') as f:
    root = f["ninapro_db5"]
    for subj_key in root.keys():
        print(f"Subject: {subj_key}")
        subj_group = root[subj_key]
        print(subj_group.attrs.keys())  # Print static metadata
        print(subj_group.keys())  # List all samples for this subject
        for sample_key in subj_group.keys():
            print(f"  Sample: {sample_key}")
            sample_group = subj_group[sample_key]
            print(sample_group.attrs.keys())  # Print per-sample metadata
            emg_data = sample_group["emg"][:]
            accel_data = sample_group["accel"][:]
            glove_data = sample_group["glove"][:]
            print(f"    EMG shape: {emg_data.shape}")
            print(f"    Accel shape: {accel_data.shape}")
            print(f"    Glove shape: {glove_data.shape}")
            

Subject: S1
<KeysViewHDF5 ['age', 'circumference_cm', 'gender', 'height_cm', 'laterality', 'weight_kg']>
<KeysViewHDF5 ['E1_R1_G1', 'E1_R1_G10', 'E1_R1_G11', 'E1_R1_G12', 'E1_R1_G13', 'E1_R1_G14', 'E1_R1_G15', 'E1_R1_G16', 'E1_R1_G17', 'E1_R1_G2', 'E1_R1_G3', 'E1_R1_G4', 'E1_R1_G5', 'E1_R1_G6', 'E1_R1_G7', 'E1_R1_G8', 'E1_R1_G9', 'E1_R2_G1', 'E1_R2_G10', 'E1_R2_G11', 'E1_R2_G12', 'E1_R2_G13', 'E1_R2_G14', 'E1_R2_G15', 'E1_R2_G16', 'E1_R2_G17', 'E1_R2_G2', 'E1_R2_G3', 'E1_R2_G4', 'E1_R2_G5', 'E1_R2_G6', 'E1_R2_G7', 'E1_R2_G8', 'E1_R2_G9', 'E1_R3_G1', 'E1_R3_G10', 'E1_R3_G11', 'E1_R3_G12', 'E1_R3_G13', 'E1_R3_G14', 'E1_R3_G15', 'E1_R3_G16', 'E1_R3_G17', 'E1_R3_G2', 'E1_R3_G3', 'E1_R3_G4', 'E1_R3_G5', 'E1_R3_G6', 'E1_R3_G7', 'E1_R3_G8', 'E1_R3_G9', 'E1_R4_G1', 'E1_R4_G10', 'E1_R4_G11', 'E1_R4_G12', 'E1_R4_G13', 'E1_R4_G14', 'E1_R4_G15', 'E1_R4_G16', 'E1_R4_G17', 'E1_R4_G2', 'E1_R4_G3', 'E1_R4_G4', 'E1_R4_G5', 'E1_R4_G6', 'E1_R4_G7', 'E1_R4_G8', 'E1_R4_G9', 'E1_R5_G1', 'E1_R5_G10', 'E1_R5_